In [1]:
print("Basics of Langgraph")

Basics of Langgraph


In [2]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [3]:
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

load_dotenv()

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=MODEL,
)

## Using Groq Instead of OpenAI 

In [4]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")  #

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),  #
    model=MODEL,  #
)


In [5]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.0'}} profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000002293DE5C090> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002293DF84D50> model_name='openai/gpt-oss-20b' model_kwargs={} groq_api_key=SecretStr('**********')


In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

In [7]:

def ask(system_prompt: str, user_prompt: str) -> str:
    if not os.getenv("GROQ_API_KEY"):
        raise ValueError("OPENAI_API_KEY is not set")
    return llm.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ]
    ).content

In [8]:
ask("You are a helpful assistant","What is the capital of India")

'The capital of India is **New Delhi**.'

## State

In [9]:
from typing_extensions import TypedDict

In [10]:
class TicketState(TypedDict):
    text: str 
    normalized: str
    category: str

## Nodes

In [11]:
def normalize(s: TicketState) -> dict:
    return {"normalized": " ".join(s["text"].lower().split())}

In [12]:
def classify(s: TicketState) -> dict:
    category = ask(
        "Classify as billing, technical, support or other. Output one lowercase word.", s["normalized"].strip().lower()
    )
    return {"category": category if category in ["billing", "technical", "support", "other"] else "other"}

## Create the graph

In [13]:
from langgraph.graph import StateGraph, START, END

In [14]:
builder = StateGraph(TicketState)

builder.add_node("normalize", normalize)
builder.add_node("classify", classify)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "classify")
builder.add_edge("classify", END)

## Compile the graph

In [15]:
graph = builder.compile()

graph.invoke({"text": "I need to cancel my subscription"}) # running the graph

{'text': 'I need to cancel my subscription',
 'normalized': 'i need to cancel my subscription',
 'category': 'billing'}

In [16]:
from langchain_core.runnables import RunnableConfig
from typing_extensions import Annotated, TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langgraph.runtime import Runtime

def reducer(a: list, b: int | None) -> list:
    if b is not None:
        return a + [b]
    return a

class State(TypedDict):
    x: Annotated[list[float], reducer]

class Context(TypedDict):
    r: float

graph = StateGraph(state_schema=State, context_schema=Context)

def node(state: State, runtime: Runtime[Context]) -> dict:
    r = runtime.context.get("r", 1.0)
    x = state["x"][-1] # last element of the list
    next_value = x * r * (1 - x) # 0.75
    
    return {"x": next_value}

graph.add_node("A", node)
graph.set_entry_point("A")
graph.set_finish_point("A")
compiled = graph.compile()

step1 = compiled.invoke({"x": 0.5}, context={"r": 3.0})
# {'x': [0.5, 0.75]}

In [17]:
step1

{'x': [0.5, 0.75]}